# Ontology-Constrained Memory (OCM) — Google Colab runner

A write-time **governed** memory layer for long-horizon LLM agents. This notebook runs the project end to end:

1. Get the code & install deps
2. Sanity tests
3. **Offline governance demo** (no GPU, no API key)
4. Benchmark + metrics (baselines B0–B3)
5. Full experiment suite (multi-seed CIs, significance, τ-sweep, stress)
6. *(optional)* Real embeddings (`sentence-transformers`)
7. *(optional)* Local **Qwen** extractor via vLLM

Sections 1–5 run on a **CPU-only** runtime. Section 7 needs a **GPU** runtime (Runtime → Change runtime type → GPU).

## 1. Get the code

**Option A — clone from GitHub** (edit `REPO_URL`). **Option B** (next cell) — upload a zip or mount Google Drive.

In [ ]:
# Option A: clone from a Git remote (edit this URL to your fork/repo).
import os
REPO_URL = "https://github.com/<your-username>/ocmr.git"  # <-- EDIT ME
REPO_DIR = "/content/ocmr"
if not os.path.exists(REPO_DIR):
    rc = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if rc != 0:
        print("Clone failed — use Option B (upload/Drive) below.")
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    print("cwd:", os.getcwd())

In [ ]:
# Option B (only if you did NOT clone): upload a zip of the project, or mount Drive.
#
# --- B1: upload a zip named ocmr.zip that contains the repo root (with ocm/, requirements.txt) ---
# from google.colab import files
# up = files.upload()                      # choose ocmr.zip
# !unzip -q ocmr.zip -d /content && ls /content
# %cd /content/ocmr                        # adjust if the zip nests a folder
#
# --- B2: mount Google Drive and cd to where you stored the repo ---
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/ocmr          # adjust path

## 2. Install dependencies

The core (offline) demo needs only a few light packages. `chromadb` and `sentence-transformers` are **optional** — OCM falls back to a pure-Python vector index and a deterministic embedding provider when they are absent.

In [ ]:
# Light core install (enough for sections 2–5).
!pip -q install "pydantic>=2.6,<3" "networkx>=3.2" "fastapi>=0.110" "uvicorn>=0.29" "httpx>=0.27" "pytest>=8.0" "hypothesis>=6.100"

# Make the ocm package importable from the repo root.
import sys, os
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import ocm
print("OCM importable from", ROOT)

## 3. Sanity tests

Run a fast slice of the suite (the full suite is ~450 tests and also passes).

In [ ]:
!python -m pytest -q ocm/tests/test_has_status_assertions.py ocm/tests/test_experiment_stats.py ocm/tests/test_llm_realembeddings_wiring.py

## 4. Offline governance demo (no GPU, no API key)

Watch write-time governance: facts are accepted, a status flip is **quarantined** (not silently overwritten), and a correction **supersedes**. The status query then surfaces the contradiction inline.

In [ ]:
from ocm.core.config import Settings
from ocm.core.container import CoreContainer

c = CoreContainer(Settings(deterministic_test_mode=True, chroma_mode="memory", extractor="mock"))

def show(r, label):
    print(f"\n# {label}")
    print("  accepted   :", [o.candidate.predicate for o in r.accepted])
    print("  superseded :", [o.candidate.predicate for o in r.superseded])
    print("  quarantined:", [o.reason for o in r.quarantined])

show(c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1"), "W1 ownership + assignment")
show(c.write_pipeline.run("Bob completed Task T1.", "s2"), "W2 completion -> T1 done")
show(c.write_pipeline.run("Task T1 is not started.", "s3"), "W3 status flip -> QUARANTINED")
show(c.write_pipeline.run("Actually, Carol is assigned to Task T1.", "s4"), "W4 correction -> SUPERSEDE")

pkg = c.retrieval_pipeline.query("What is the current status of Task T1?", top_k=10)
print("\nQuery: 'What is the current status of Task T1?'")
print("  answer   :", pkg.answer)
print("  conflicts:", [{"accepted": cf.accepted, "quarantined": cf.quarantined, "reason": cf.reason} for cf in pkg.conflicts])

## 5a. Benchmark + metrics (baselines B0–B3)

Generates the seeded benchmark and reports retrieval / answer / write-time / agent metrics with deltas vs B0.

In [ ]:
!python -m ocm.scripts.report_metrics --seed 1337

## 5b. Full experiment suite (multi-seed CIs, significance, τ-sweep, stress)

`--quick` is a fast smoke run. For the full protocol drop `--quick` and pass 5 seeds (slower). These are the *system's own* offline numbers (deterministic mock pipeline), not the paper's illustrative figures.

In [ ]:
!python -m ocm.scripts.run_experiments --quick

# Full protocol (slower):
# !python -m ocm.scripts.run_experiments --seeds 1337 7 42 99 2024 --per-category 6 --out results.json

## 6. (Optional) Real embeddings

Swaps the deterministic hashing embeddings for the real `all-MiniLM-L6-v2` model. First run downloads the weights (~90 MB). Runs on CPU or GPU.

In [ ]:
!pip -q install "sentence-transformers>=2.6"

from ocm.core.config import Settings
from ocm.core.container import CoreContainer

# deterministic_test_mode=False -> the container loads LocalEmbeddingProvider (real MiniLM).
# sqlite_path=':memory:' keeps storage hermetic (no files written).
s = Settings(deterministic_test_mode=False, extractor="mock", embedding_mode="local",
             sqlite_path=":memory:", chroma_mode="memory")
c = CoreContainer(s)
c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1")
print("owner answer:", c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5).answer)

## 7. (Optional) Local Qwen extractor via vLLM — needs a GPU runtime

OCM's LLM extractor speaks the **OpenAI-compatible** `/chat/completions` API, so any local server works.

**Model vs hardware:**
- Free Colab **T4 (16 GB)** → use `Qwen/Qwen2.5-7B-Instruct`.
- Colab Pro+ **A100 (40 GB)** → `Qwen/Qwen2.5-32B-Instruct-AWQ` (4-bit) fits; full bf16 `Qwen/Qwen2.5-32B-Instruct` needs ~2× 80 GB and will **not** fit a single Colab GPU.

Set `MODEL` below to match your GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU — switch Runtime type to GPU."

In [ ]:
!pip -q install vllm

import subprocess, time, os, urllib.request

MODEL = "Qwen/Qwen2.5-7B-Instruct"          # T4. For A100: "Qwen/Qwen2.5-32B-Instruct-AWQ"
PORT = 8000
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", str(PORT), "--max-model-len", "8192"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT,
)
print("starting vLLM (first run downloads the model)...")
ok = False
for _ in range(180):  # up to ~15 min for a cold model download
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=3)
        ok = True; break
    except Exception:
        time.sleep(5)
print("vLLM ready" if ok else "vLLM not ready yet — tail vllm.log")
if not ok:
    !tail -n 30 vllm.log

In [ ]:
# Point OCM at the local Qwen server and extract with the real LLM.
import os
os.environ["OCM_LLM_API_KEY"] = "local"   # vLLM ignores the key; any non-empty value
from ocm.core.config import Settings
from ocm.core.container import CoreContainer

s = Settings(
    extractor="llm",
    llm_base_url=f"http://localhost:{PORT}/v1",
    llm_model=MODEL,
    llm_api_key="local",
    llm_use_json_mode=True,           # vLLM supports JSON mode; set False for servers that reject it
    deterministic_test_mode=True,     # cheap deterministic embeddings + in-memory storage (LLM still used for W1)
    chroma_mode="memory",
)
c = CoreContainer(s)
r = c.write_pipeline.run("Alice owns Project Orion. Bob is assigned to Task T1.", "s1")
print("accepted:", [o.candidate.predicate for o in r.accepted])
print("owner answer:", c.retrieval_pipeline.query("Who owns Project Orion?", top_k=5).answer)

In [ ]:
# Run the experiment suite using the local Qwen extractor (quick smoke).
!python -m ocm.scripts.run_experiments --quick \
  --extractor llm --llm-base-url http://localhost:8000/v1 --llm-model "$MODEL" \
  --embeddings deterministic

### Notes & caveats
- Sections 3–5 are fully offline/deterministic and need no GPU or keys.
- `--embeddings local` and the Qwen section produce genuine stochasticity (random ids, real models) — that's what makes multi-seed CIs and significance tests meaningful.
- Storage stays in-memory in every demo, so nothing is written to disk except `benchmark.jsonl` / `results.json` when you ask for them.
- To stop the vLLM server: `server.terminate()`.
- `Qwen2.5-32B-Instruct` (bf16) will not fit a single Colab GPU; use the `-AWQ` 4-bit build on A100, or `Qwen2.5-7B-Instruct` on T4.